# Assignment: Build and Test a Machine-Learned Interatomic Potential



### Instructions
1. Complete every cell marked `# TODO`. Do **not** change the names or signatures of the functions you are asked to write, because the check cells call them.
2. After each task, run the **check cell** that follows it. A check prints `PASSED` when your implementation is correct. Passing checks is necessary but not sufficient for full marks: plots, tables and written answers are graded too.
3. Only NumPy and Matplotlib are allowed. Do not import ASE, PyTorch, scikit-learn or any MLIP package.
4. Use the practice-session notebook and the class notes as references. Copying code from the practice notebook is allowed **only** where it is marked as provided here. Everything else must be written by you.
5. Submit the executed notebook (`.ipynb`) with all outputs visible.

### The problem
You will build a linear MLIP for an 8-atom cluster described by a reference potential that differs from the one in the practice session: a **Morse** pair potential plus a three-body term that favours **90°** bond angles. All quantities are in reduced units ($D_e=1$, $r_0=1$, $m=1$, $k_B=1$).


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
plt.rcParams.update({"figure.dpi": 100, "font.size": 10, "axes.grid": True, "grid.alpha": 0.3})

def check(cond, msg=""):
    print("PASSED" if cond else f"FAILED  {msg}")

# ---- provided helpers (same as in the practice session) ----
def pair_geometry(R):
    D = R[None, :, :] - R[:, None, :]          # D[i, j] = r_j - r_i
    return D, np.linalg.norm(D, axis=-1)

def numerical_forces(energy_fn, R, h=1e-5):
    F = np.zeros_like(R)
    for i in range(R.shape[0]):
        for a in range(3):
            Rp = R.copy(); Rp[i, a] += h
            Rm = R.copy(); Rm[i, a] -= h
            F[i, a] = -(energy_fn(Rp) - energy_fn(Rm)) / (2 * h)
    return F

def fc_cos(r, rc):
    return np.where(r < rc, 0.5 * (np.cos(np.pi * np.minimum(r, rc) / rc) + 1.0), 0.0)

def random_rotation(rng):
    Q, Rq = np.linalg.qr(rng.normal(size=(3, 3)))
    Q = Q @ np.diag(np.sign(np.diag(Rq)))
    if np.linalg.det(Q) < 0:
        Q[:, 0] *= -1
    return Q

rng = np.random.default_rng(2026)

---
## Task A. Morse potential: energy and analytic forces
$$
V(r)=D_e\Big[\big(1-e^{-a(r-r_0)}\big)^2-1\Big],\qquad E_{\rm Morse}=\sum_{i<j}V(r_{ij})
$$

with $D_e=1$, $a=3$, $r_0=1$.

**A1 (3 pts).** Show by hand (write it in the markdown cell below) that $V(r_0)=-D_e$ and that

$$
V'(r)=2aD_e\,e^{-a(r-r_0)}\big(1-e^{-a(r-r_0)}\big).
$$

**A2 (7 pts).** Implement `morse_energy(R)` and `morse_forces(R)`. The force must be **analytic** (use $\mathbf F_i=\sum_{j\ne i}V'(r_{ij})\,\mathbf d_{ij}/r_{ij}$ with $\mathbf d_{ij}=\mathbf r_j-\mathbf r_i$). Do not call `numerical_forces` inside `morse_forces`.

**A1.  derivation:**

Put $r = r_0$. Then $e^{-a(r-r_0)} = 1$, so

$$V(r_0) = D_e\big[(1-1)^2 - 1\big] = -D_e.$$

For the slope, let $u = e^{-a(r-r_0)}$. Then $V = D_e[(1-u)^2 - 1]$ and $du/dr = -au$.

$$
\frac{dV}{dr} = D_e \cdot 2(1-u)\cdot (-du/dr) = 2a D_e\, u(1-u)
= 2a D_e\, e^{-a(r-r_0)}\big(1 - e^{-a(r-r_0)}\big).
$$

Therefore, the derivative is zero at r = r_0, confirming that r_0 is the equilibrium bond distance and V(r_0) = -D_e.

In [2]:
D_E, A_M, R0_M = 1.0, 3.0, 1.0

def morse_energy(R):
    # each unordered pair once
    _, rij = pair_geometry(R)
    i, j = np.triu_indices(len(R), k=1)
    x = np.exp(-A_M * (rij[i, j] - R0_M))
    return np.sum(D_E * ((1.0 - x)**2 - 1.0))

def morse_forces(R):
    # F_i = sum_{j != i} V'(r_ij) * (r_j - r_i) / r_ij
    D, rij = pair_geometry(R)
    np.fill_diagonal(rij, np.inf)
    x = np.exp(-A_M * (rij - R0_M))
    dVdr = 2.0 * A_M * D_E * x * (1.0 - x)
    coef = dVdr / rij
    return np.sum(coef[:, :, None] * D, axis=1)

In [3]:
# ---- check A ----
dimer = np.array([[0.0, 0, 0], [1.0, 0, 0]])
check(np.isclose(morse_energy(dimer), -1.0), "V(r0) should be -1")
dimer2 = np.array([[0.0, 0, 0], [1.3, 0, 0]])
y = np.exp(-3 * 0.3)
check(np.isclose(morse_energy(dimer2), (1 - y)**2 - 1), "dimer energy at r = 1.3")
Rt = np.array([[i, j, k] for i in (0, 1) for j in (0, 1) for k in (0, 1)], float) + rng.normal(scale=0.08, size=(8, 3))
err = np.abs(morse_forces(Rt) - numerical_forces(morse_energy, Rt)).max()
check(err < 1e-6, f"analytic vs numerical force difference = {err:.2e}")
check(np.allclose(morse_forces(Rt).sum(0), 0, atol=1e-10), "forces must sum to zero (Newton's third law)")

PASSED
PASSED
PASSED
PASSED


---
## Task B. Polynomial envelope cutoff
Implement the envelope used by DimeNet, NequIP and MACE:

$$
f_{\rm env}(r)=1-\frac{(p+1)(p+2)}{2}x^{p}+p(p+2)\,x^{p+1}-\frac{p(p+1)}{2}x^{p+2},\qquad x=\frac{r}{r_c},
$$

and $f_{\rm env}=0$ for $r\ge r_c$. The function must accept NumPy arrays of any shape.

In [4]:
def f_env(r, rc, p=6):
    r = np.asarray(r, dtype=float)
    x = r / rc
    f = (1.0
         - ((p + 1) * (p + 2) / 2.0) * x**p
         + (p * (p + 2)) * x**(p + 1)
         - (p * (p + 1) / 2.0) * x**(p + 2))
    return np.where(r < rc, f, 0.0)

In [5]:
# ---- check B ----
rc, h = 2.0, 1e-4
check(np.isclose(f_env(0.0, rc), 1.0), "f(0) = 1")
check(abs(f_env(rc - 1e-9, rc)) < 1e-12, "f(rc) = 0")
slope = (f_env(rc - h, rc) - f_env(rc - 3*h, rc)) / (2*h)
check(abs(slope) < 1e-6, f"slope near rc = {slope:.2e}")
check(f_env(np.ones((3, 4)) * 0.5, rc).shape == (3, 4), "must work on arrays")
check(np.isclose(f_env(1.0, 2.0), 1 - 28/64 + 48/128 - 21/256), "value at x = 0.5")

PASSED
PASSED
PASSED
PASSED
PASSED


---
## Task C. Behler symmetry functions
Implement, **for every atom $i$ at once**,

$$
G_i^{2}(\eta,R_s)=\sum_{j\ne i}e^{-\eta(r_{ij}-R_s)^2}f_c(r_{ij}),
$$
$$
G_i^{4}(\eta,\zeta,\lambda)=2^{1-\zeta}\sum_{j<k;\ j,k\ne i}(1+\lambda\cos\theta_{jik})^{\zeta}\,e^{-\eta(r_{ij}^2+r_{ik}^2+r_{jk}^2)}\,f_c(r_{ij})f_c(r_{ik})f_c(r_{jk}),
$$

using the cosine cutoff `fc_cos` provided above. Note: the sum counts each unordered pair $\{j,k\}$ **once**.

- `g2_atoms(R, etas, Rss, rc)` returns an array of shape `(N, len(etas))`, where column $m$ uses $(\eta_m, R_{s,m})$.
- `g4_atoms(R, params, rc)` returns shape `(N, len(params))`, where `params` is a list of tuples `(eta, zeta, lam)`.

Loops over atoms are allowed, but vectorised code (as in the practice notebook) runs about 50 times faster and makes Tasks F to H much quicker.

In [6]:
def g2_atoms(R, etas, Rss, rc):
    _, rij = pair_geometry(R)
    etas = np.atleast_1d(np.asarray(etas, dtype=float))
    Rss = np.atleast_1d(np.asarray(Rss, dtype=float))
    fc = fc_cos(rij, rc)
    np.fill_diagonal(fc, 0.0)          # no self term
    dr = rij[:, :, None] - Rss[None, None, :]
    g = np.exp(-etas * dr**2) * fc[:, :, None]
    return np.sum(g, axis=1)

def g4_atoms(R, params, rc):
    D, rij = pair_geometry(R)
    N = len(R)
    np.fill_diagonal(rij, np.inf)
    fc = fc_cos(rij, rc)
    u = D / rij[:, :, None]
    cos = np.clip(np.einsum("ijx,ikx->ijk", u, u), -1.0, 1.0)
    # sum each unordered pair {j, k} once
    j_lt_k = np.triu(np.ones((N, N), dtype=bool), k=1)
    idx = np.arange(N)
    out = np.zeros((N, len(params)))
    for m, (eta, zeta, lam) in enumerate(params):
        ang = (1.0 + lam * cos)**zeta
        rad = np.exp(-eta * (rij[:, :, None]**2 + rij[:, None, :]**2 + rij[None, :, :]**2))
        fcprod = fc[:, :, None] * fc[:, None, :] * fc[None, :, :]
        term = (2.0**(1 - zeta)) * ang * rad * fcprod
        term[:, idx, idx] = 0.0
        term[idx, idx, :] = 0.0
        term[idx, :, idx] = 0.0
        term = np.where(j_lt_k[None, :, :], term, 0.0)
        out[:, m] = term.sum(axis=(1, 2))
    return out

In [7]:
# ---- check C: Example 3 of the class notes ----
th = np.deg2rad(104.5)
R3 = np.array([[0, 0, 0], [1.0, 0, 0], [1.5*np.cos(th), 1.5*np.sin(th), 0]])
G2 = g2_atoms(R3, np.array([0.5]), np.array([0.0]), 5.0)
G4 = g4_atoms(R3, [(0.1, 1, 1), (0.1, 1, -1)], 5.0)
check(G2.shape == (3, 1) and G4.shape == (3, 2), "output shapes")
check(np.isclose(G2[0, 0], 0.80635, atol=1e-4), f"G2 of atom 0 = {G2[0,0]:.5f}, expected 0.80635")
check(np.allclose(G4[0], [0.17059, 0.28455], atol=1e-4), f"G4 of atom 0 = {G4[0]}, expected [0.17059 0.28455]")
# a second test with more atoms, against a slow but obvious reference implementation
def g4_slow(R, eta, zeta, lam, rc):
    N = len(R); out = np.zeros(N)
    for i in range(N):
        for j in range(N):
            for k in range(j + 1, N):
                if i in (j, k): continue
                a, b = R[j] - R[i], R[k] - R[i]; rij, rik, rjk = np.linalg.norm(a), np.linalg.norm(b), np.linalg.norm(R[k] - R[j])
                c = a @ b / (rij * rik)
                out[i] += 2**(1-zeta) * (1 + lam*c)**zeta * np.exp(-eta*(rij**2 + rik**2 + rjk**2)) * fc_cos(rij, rc) * fc_cos(rik, rc) * fc_cos(rjk, rc)
    return out
check(np.allclose(g4_atoms(Rt, [(0.5, 2, -1)], 2.0)[:, 0], g4_slow(Rt, 0.5, 2, -1, 2.0)), "G4 on the 8-atom test cube")

PASSED
PASSED
PASSED
PASSED


---
## Task D. Invariance tests
**D1 (6 pts).** Write `invariance_errors(desc_fn, R, rng)` that applies (i) a random rotation, (ii) a reflection $z\to-z$, (iii) a random translation and (iv) a random permutation of atoms to the configuration `R`. The function must return a dictionary with keys `"rotation"`, `"reflection"`, `"translation"`, `"permutation"` and, as values, the maximum absolute difference between the descriptors before and after each operation. Remember that a permutation reorders the rows of the descriptor matrix.

**D2 (4 pts).** A student proposes the "descriptor" $\mathbf q_i=\sum_{j\ne i}(\mathbf r_j-\mathbf r_i)\,f_c(r_{ij})$, a 3-component vector per atom. Use your function to test it and explain the result in one or two sentences. Which category of the notes (invariant or equivariant) does it belong to?

In [8]:
def invariance_errors(desc_fn, R, rng):
    G0 = desc_fn(R)

    Q = random_rotation(rng)
    R_rot = (Q @ R.T).T

    R_refl = R.copy()
    R_refl[:, 2] *= -1

    R_shift = R + rng.normal(size=3)

    order = rng.permutation(len(R))
    G_perm = desc_fn(R[order])

    def max_diff(A, B):
        return float(np.max(np.abs(A - B)))

    return {
        "rotation": max_diff(desc_fn(R_rot), G0),
        "reflection": max_diff(desc_fn(R_refl), G0),
        "translation": max_diff(desc_fn(R_shift), G0),
        "permutation": max_diff(G_perm, G0[order]),
    }

def student_vector_descriptor(R, rc=2.5):
    D, r = pair_geometry(R)
    return np.sum(D * fc_cos(r, rc)[:, :, None], axis=1)

vec_errs = invariance_errors(student_vector_descriptor, Rt, np.random.default_rng(1))
print("student vector descriptor:")
for name, val in vec_errs.items():
    print(f"  {name:12s} {val:.4e}")

student vector descriptor:
  rotation     3.6015e+00
  reflection   3.5314e+00
  translation  4.4409e-16
  permutation  2.2204e-16


In [9]:
# ---- check D1 ----
desc_test = lambda X: np.hstack([g2_atoms(X, np.full(6, 4.0), np.linspace(0.8, 2.2, 6), 2.5),
                                 g4_atoms(X, [(0.5, 1, 1), (0.5, 4, -1)], 2.5)])
errs = invariance_errors(desc_test, Rt, np.random.default_rng(0))
print({k: f"{v:.1e}" for k, v in errs.items()})
check(set(errs) == {"rotation", "reflection", "translation", "permutation"}, "dictionary keys")
check(max(errs.values()) < 1e-10, "G2/G4 must be invariant under all four operations")
bad = lambda X: X.copy()   # raw coordinates as a 'descriptor'
e_bad = invariance_errors(bad, Rt, np.random.default_rng(0))
check(e_bad["rotation"] > 1e-3 and e_bad["translation"] > 1e-3, "your function must detect non-invariant descriptors")

{'rotation': '1.8e-15', 'reflection': '0.0e+00', 'translation': '1.3e-15', 'permutation': '8.9e-16'}
PASSED
PASSED
PASSED


**D2. Answer:**

The rotation error is about 3.6 and the reflection error is about 3.5, while translation and permutation are at roundoff. So $\mathbf q_i$ changes as soon as the cluster is turned or mirrored. It is a sum of the vectors $(\mathbf r_j-\mathbf r_i)$, and those vectors rotate with the coordinates, which is equivariance rather than invariance. A scalar energy needs an invariant input, so this descriptor cannot be dropped straight into $E = \mathbf w\cdot\mathbf G$.

---
## Task E. The reference potential and dataset

The reference potential (provided) is

$$
E_{\rm ref}=E_{\rm Morse}+\frac{K}{2}\sum_i\sum_{j\ne k}\big(\cos\theta_{jik}-c_0\big)^2\,g(r_{ij})\,g(r_{ik}),\qquad K=0.6,\ c_0=0,\ g(r)=f_c^{\cos}(r;1.4).
$$

With $c_0=0$ the three-body term favours right angles. The cell below relaxes a cube of 8 atoms and samples 360 configurations with Langevin MD at $T=0.05,\ 0.15,\ 0.25$. **Run it as given** (it uses your Morse functions and takes about 30 s).

In [ ]:
# ---- provided: reference potential, relaxation, Langevin sampling ----
def three_body_energy(R, K=0.6, c0=0.0, rc3=1.4):
    D, r = pair_geometry(R); N = len(R)
    np.fill_diagonal(r, np.inf)
    g = fc_cos(r, rc3); u = D / r[:, :, None]
    cos = np.einsum("ijx,ikx->ijk", u, u)
    t = (cos - c0)**2 * g[:, :, None] * g[:, None, :]
    idx = np.arange(N); t[:, idx, idx] = 0.0
    return 0.5 * K * t.sum()

def ref_energy(R): return morse_energy(R) + three_body_energy(R)
def ref_forces(R): return morse_forces(R) + numerical_forces(three_body_energy, R)

def relax(R, efun, ffun, fmax=1e-4, maxit=5000):
    R = R.copy(); step = 0.01; E = efun(R)
    for _ in range(maxit):
        F = ffun(R)
        if np.abs(F).max() < fmax: break
        Rn = R + step * F; En = efun(Rn)
        if En < E: R, E = Rn, En; step *= 1.1
        else: step *= 0.5
    return R

def langevin(R, ffun, T, nsteps, dt=0.005, gamma=1.0, every=40, rng=None):
    R = R.copy(); v = rng.normal(scale=np.sqrt(T), size=R.shape); v -= v.mean(0)
    F = ffun(R); c1 = np.exp(-gamma * dt); c2 = np.sqrt((1 - c1**2) * T); frames = []
    for s in range(nsteps):
        v += 0.5 * dt * F; R += 0.5 * dt * v
        v = c1 * v + c2 * rng.normal(size=R.shape)
        R += 0.5 * dt * v; F = ffun(R); v += 0.5 * dt * F
        if (s + 1) % every == 0: frames.append(R.copy())
    return frames

cube = np.array([[i, j, k] for i in (0, 1) for j in (0, 1) for k in (0, 1)], float)
R_min = relax(cube, ref_energy, ref_forces)
N_at = len(R_min)
print(f"relaxed cluster energy: {ref_energy(R_min):.4f}")

t0 = time.time(); rng_md = np.random.default_rng(11)
configs, energies, forces, temps = [], [], [], []
for T in (0.05, 0.15, 0.25):
    for X in langevin(R_min, ref_forces, T, 4800, rng=rng_md):
        configs.append(X); energies.append(ref_energy(X)); forces.append(ref_forces(X)); temps.append(T)
configs, energies, forces, temps = map(np.array, (configs, energies, forces, temps))
print(f"{len(configs)} configurations generated in {time.time()-t0:.1f} s")

**E1 (5 pts).** (a) Plot histograms of the energy per atom for the three temperatures in one figure. (b) Print the shortest and longest interatomic distance found in the whole dataset, and the largest distance of any atom from the cluster centre of mass. (c) In one sentence: did the cluster stay intact at $T=0.25$? Justify with your numbers.

In [ ]:
# (a) energy per atom at each temperature
fig, ax = plt.subplots(figsize=(6.4, 4.0))
for T, col in ((0.05, "C0"), (0.15, "C1"), (0.25, "C2")):
    e_atom = energies[temps == T] / N_at
    ax.hist(e_atom, bins=16, alpha=0.55, color=col, label=f"T = {T}")
ax.set_xlabel("energy per atom")
ax.set_ylabel("number of frames")
ax.set_title("Sampled energies")
ax.legend()
fig.tight_layout()
plt.show()

# (b) pair distances and distance from the centre of mass
def pair_minmax(R):
    _, rij = pair_geometry(R)
    np.fill_diagonal(rij, np.inf)
    return float(rij.min()), float(np.max(rij[np.isfinite(rij)]))

r_short, r_long = np.inf, 0.0
r_from_com = 0.0
for X in configs:
    a, b = pair_minmax(X)
    r_short = min(r_short, a)
    r_long = max(r_long, b)
    dcom = np.linalg.norm(X - X.mean(axis=0), axis=1).max()
    r_from_com = max(r_from_com, float(dcom))

print(f"shortest interatomic distance : {r_short:.4f}")
print(f"longest interatomic distance  : {r_long:.4f}")
print(f"largest distance from COM     : {r_from_com:.4f}")

print("same three numbers, split by temperature:")
for T in (0.05, 0.15, 0.25):
    rs, rl, rcmax = np.inf, 0.0, 0.0
    for X in configs[temps == T]:
        a, b = pair_minmax(X)
        rs, rl = min(rs, a), max(rl, b)
        rcmax = max(rcmax, float(np.linalg.norm(X - X.mean(0), axis=1).max()))
    print(f"  T = {T:.2f}   rmin = {rs:.3f}   rmax = {rl:.3f}   max |r-com| = {rcmax:.3f}")

**E1 (c). answer:**

Yes, the cluster stays in one piece at $T=0.25$: the farthest atom is only 1.37 from the centre of mass (1.05 at $T=0.05$) and the longest pair is 2.40, so no atom has left.

---
## Task F. A linear MLIP trained on energies and forces

Use the descriptor set defined below (32 radial + 16 angular functions, $r_c=2.5$) and the **provided** Jacobian routine.

The model is $E=Nb+\mathbf w\cdot\sum_i\tilde{\mathbf G}_i$ with standardised descriptors $\tilde G_{i\mu}=(G_{i\mu}-\mu_\mu)/s_\mu$, and forces $\mathbf F_k=-\sum_\mu w_\mu\,\partial(\sum_i\tilde G_{i\mu})/\partial\mathbf r_k$.

**F1 (10 pts).** Implement
- `fit_linear_mlip(idx, cols, use_forces=True, wF=1.0, ridge=1e-8)`: builds the least-squares system described in §7.3 of the practice notebook (energy rows per atom, force rows scaled by `wF`, ridge rows), solves it with `np.linalg.lstsq`, and returns a dictionary with keys `"b"`, `"w"`, `"mu"`, `"sd"`, `"cols"`. Here `idx` are training indices and `cols` the descriptor columns to use. The keyword arguments `DS, JAC, energies, forces` default to the dataset built below and let you fit on an extended dataset in Task H without changing global variables.
- `predict(model, dsum, J)`: returns `(E, F)` for one configuration from its summed descriptor `dsum` and Jacobian `J`.
- `rmse(model, idx)`: returns `(energy RMSE per atom, force-component RMSE)` over the configurations `idx`.

**F2 (10 pts).** Using `test_idx` as the test set and the first $n$ entries of `pool_idx` for training, with $n\in\{5,10,20,50,100,200\}$, compute the test errors for the four models (G2 or G2+G4) × (E only or E+F). Print a table and plot both learning curves on log–log axes. Then answer the questions in the markdown cell that follows.

In [ ]:
# ---- provided: descriptor set and Jacobians ----
RC = 2.5
ETAS = np.full(32, 16.0); RSS = np.linspace(0.6, 2.4, 32)
G4P = [(eta, zeta, lam) for eta in (0.3, 1.5) for zeta in (1, 2, 4, 16) for lam in (1, -1)]
N_G2, N_G4 = len(ETAS), len(G4P)
cols_G2 = np.arange(N_G2); cols_all = np.arange(N_G2 + N_G4)

def descriptors(R):
    return np.hstack([g2_atoms(R, ETAS, RSS, RC), g4_atoms(R, G4P, RC)])

def descriptor_sum_and_jacobian(R, desc_fn=descriptors, h=1e-5):
    d0 = desc_fn(R).sum(axis=0); J = np.zeros((d0.size,) + R.shape)
    for k in range(R.shape[0]):
        for a in range(3):
            Rp = R.copy(); Rp[k, a] += h; Rm = R.copy(); Rm[k, a] -= h
            J[:, k, a] = (desc_fn(Rp).sum(0) - desc_fn(Rm).sum(0)) / (2 * h)
    return d0, J

t0 = time.time()
DS, JAC = map(np.array, zip(*[descriptor_sum_and_jacobian(X) for X in configs]))
print(f"DS {DS.shape}, JAC {JAC.shape}, {time.time()-t0:.1f} s")
perm_all = np.random.default_rng(0).permutation(len(configs))
test_idx, pool_idx = perm_all[:100], perm_all[100:]

In [ ]:
def fit_linear_mlip(idx, cols, use_forces=True, wF=1.0, ridge=1e-8, DS=DS, JAC=JAC, energies=energies, forces=forces):
    idx = np.asarray(idx)
    cols = np.asarray(cols)
    N = JAC.shape[2]
    # standardise the per-atom average descriptor on the training set
    g_mean = DS[idx][:, cols] / N
    mu = g_mean.mean(axis=0)
    sd = g_mean.std(axis=0)
    sd = np.where(sd < 1e-12, 1.0, sd)

    M = len(cols)
    # one energy row per configuration, written per atom: E/N = b + w . g_tilde
    A_e = np.zeros((len(idx), 1 + M))
    A_e[:, 0] = 1.0
    A_e[:, 1:] = (g_mean - mu) / sd
    y_e = energies[idx] / N

    blocks_A = [A_e]
    blocks_y = [y_e]

    if use_forces:
        # F_k = - sum_mu w_mu * d(sum_i Gtilde)/dr_k
        J = JAC[idx][:, cols]                     # (n, M, N, 3)
        J = np.moveaxis(J, 1, -1).reshape(-1, M)  # (n*N*3, M)
        A_f = np.zeros((J.shape[0], 1 + M))
        A_f[:, 1:] = -(J / sd) * wF
        y_f = forces[idx].reshape(-1) * wF
        blocks_A.append(A_f)
        blocks_y.append(y_f)

    # ridge on the weights, not on the bias
    A_r = np.zeros((M, 1 + M))
    A_r[:, 1:] = np.eye(M) * np.sqrt(ridge)
    blocks_A.append(A_r)
    blocks_y.append(np.zeros(M))

    A = np.vstack(blocks_A)
    y = np.concatenate(blocks_y)
    theta, *_ = np.linalg.lstsq(A, y, rcond=None)
    return {"b": float(theta[0]), "w": theta[1:], "mu": mu, "sd": sd, "cols": cols}

def predict(model, dsum, J):
    cols = model["cols"]
    mu, sd, w, b = model["mu"], model["sd"], model["w"], model["b"]
    N = J.shape[1]
    # E = N b + w . (sum_i G_i - N mu) / sd
    E = N * b + np.dot(w, (dsum[cols] - N * mu) / sd)
    F = -np.einsum("m,mka->ka", w / sd, J[cols])
    return E, F

def rmse(model, idx, DS=DS, JAC=JAC, energies=energies, forces=forces):
    idx = np.asarray(idx)
    N = JAC.shape[2]
    e_err = []
    f_err = []
    for i in idx:
        Ep, Fp = predict(model, DS[i], JAC[i])
        e_err.append((Ep - energies[i]) / N)
        f_err.append(Fp - forces[i])
    e_rmse = np.sqrt(np.mean(np.square(e_err)))
    f_rmse = np.sqrt(np.mean(np.square(f_err)))
    return e_rmse, f_rmse

In [ ]:
# ---- check F1 ----
m_chk = fit_linear_mlip(pool_idx[:50], cols_all, use_forces=True)
check(set(m_chk) >= {"b", "w", "mu", "sd", "cols"}, "model dictionary keys")
E_chk, F_chk = predict(m_chk, DS[test_idx[0]], JAC[test_idx[0]])
check(np.shape(F_chk) == (N_at, 3), "force shape")
check(np.allclose(F_chk.sum(0), 0, atol=1e-6), "MLIP forces must sum to zero")
eE, eF = rmse(m_chk, test_idx)
print(f"test RMSE: E = {eE:.2e} per atom, F = {eF:.3f}   (force std = {forces.std():.2f})")
check(eF < 0.1 * forces.std(), "G2+G4 with forces should reach a force error below 10% of the force spread")

In [ ]:
train_sizes = [5, 10, 20, 50, 100, 200]
models_f2 = [
    ("G2, E only", cols_G2, False),
    ("G2, E+F", cols_G2, True),
    ("G2+G4, E only", cols_all, False),
    ("G2+G4, E+F", cols_all, True),
]

curve = {name: [] for name, _, _ in models_f2}
header = f"{'model':<18} {'n':>5} {'E rmse/atom':>14} {'F rmse':>10}"
print(header)
print("-" * len(header))
for n in train_sizes:
    train = pool_idx[:n]
    for name, cols, use_f in models_f2:
        m = fit_linear_mlip(train, cols, use_forces=use_f)
        e_err, f_err = rmse(m, test_idx)
        curve[name].append((e_err, f_err))
        print(f"{name:<18} {n:5d} {e_err:14.4e} {f_err:10.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.1))
for name, _, _ in models_f2:
    ee = [p[0] for p in curve[name]]
    ff = [p[1] for p in curve[name]]
    axes[0].loglog(train_sizes, ee, marker="o", label=name)
    axes[1].loglog(train_sizes, ff, marker="o", label=name)
axes[0].set_xlabel("number of training configurations")
axes[0].set_ylabel("energy RMSE per atom")
axes[0].set_title("Energy")
axes[1].set_xlabel("number of training configurations")
axes[1].set_ylabel("force-component RMSE")
axes[1].set_title("Forces")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

**F2 questions.** (a) How many training configurations does the G2+G4, E+F model need to beat the force error of the G2+G4, E-only model trained on 200 configurations? (b) Why does the G2-only model stop improving? Relate your answer to the form of $E_{\rm ref}$. (c) Count the number of scalar labels (energies plus force components) in 10 configurations and compare it with the number of model parameters.

(a) 10 configurations. The G2+G4 energy-only model trained on 200 frames has a force RMSE of 0.024. With forces in the fit, the same descriptors already reach 0.017 at $n=10$. At $n=5$ the force error is still 0.15, so 5 is not enough and 10 is the smallest size in the list that wins.

(b) G2 only sees distances. The Morse part of $E_{\rm ref}$ is a pair potential, so a radial descriptor can learn it, but the extra term is $\propto(\cos\theta - 0)^2$ and that depends on the bond angle. No amount of extra radial data represents an angle. That is why the G2 force error flattens near 0.08 while G2+G4 keeps going down to about 0.011.

(c) Ten configurations contain 10 energies and $10\times 8\times 3 = 240$ force components, 250 numbers in total. G2+G4 has 48 weights plus the bias, so 49 parameters. With the forces included, 10 frames already give far more labels than parameters. Energies alone would be only 10 labels for those 49 parameters, which is why the energy-only curves are so much worse at small $n$.

---
## Task G. Molecular dynamics with your MLIP

**G1 (7 pts).** Implement `velocity_verlet(R, ef, T0, nsteps, dt=0.005, seed=3)` where `ef(R)` returns `(E, F)`. Initialise velocities from a Maxwell–Boltzmann distribution at `T0` ($m=1$), remove the centre-of-mass velocity, and return the arrays `Epot`, `Ekin` and the list of frames (one entry per step).

**G2 (8 pts).** Train the final model (G2+G4, E+F, all 260 pool configurations). Starting from `configs[test_idx[0]]` with $T_0=0.2$, run 1500 NVE steps with the MLIP and with the reference potential. Plot the total energy of both runs versus time. Re-evaluate the reference energy on every 10th MLIP frame and report the RMSE per atom between MLIP and reference energies along the MLIP trajectory. Comment in two sentences.

In [ ]:
def velocity_verlet(R, ef, T0, nsteps, dt=0.005, seed=3):
    rng_vv = np.random.default_rng(seed)
    R = np.array(R, dtype=float, copy=True)
    # Maxwell-Boltzmann, m = 1, then take out the centre-of-mass drift
    v = rng_vv.normal(scale=np.sqrt(T0), size=R.shape)
    v -= v.mean(axis=0)

    Epot = np.empty(nsteps)
    Ekin = np.empty(nsteps)
    frames = []
    E, F = ef(R)
    for step in range(nsteps):
        v = v + 0.5 * dt * F
        R = R + dt * v
        E, F = ef(R)
        v = v + 0.5 * dt * F
        Epot[step] = E
        Ekin[step] = 0.5 * np.sum(v * v)
        frames.append(R.copy())
    return Epot, Ekin, frames

# G2 -- final model on the whole pool, then two NVE runs
model_final = fit_linear_mlip(pool_idx, cols_all, use_forces=True)
e_fin, f_fin = rmse(model_final, test_idx)
print(f"final model on the test set: E = {e_fin:.4e} per atom, F = {f_fin:.4f}")

def mlip_ef(R):
    dsum, J = descriptor_sum_and_jacobian(R)
    return predict(model_final, dsum, J)

def ref_ef(R):
    return ref_energy(R), ref_forces(R)

R0_md = configs[test_idx[0]].copy()
n_md = 1500
dt_md = 0.005

print("MLIP trajectory...")
t0 = time.time()
Ep_m, Ek_m, fr_m = velocity_verlet(R0_md, mlip_ef, 0.2, n_md, dt=dt_md, seed=3)
print(f"  {time.time() - t0:.1f} s")

print("reference trajectory...")
t0 = time.time()
Ep_r, Ek_r, fr_r = velocity_verlet(R0_md, ref_ef, 0.2, n_md, dt=dt_md, seed=3)
print(f"  {time.time() - t0:.1f} s")

t_md = np.arange(n_md) * dt_md
fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.plot(t_md, Ep_m + Ek_m, label="MLIP")
ax.plot(t_md, Ep_r + Ek_r, label="reference", alpha=0.85)
ax.ticklabel_format(useOffset=False)   # otherwise matplotlib subtracts ~16.7 and the axis looks like 0
ax.set_xlabel("time")
ax.set_ylabel("total energy")
ax.set_title("NVE from a test configuration, T0 = 0.2")
ax.legend()
fig.tight_layout()
plt.show()

every = 10
idx_sub = np.arange(0, n_md, every)
E_mlip_sub = Ep_m[idx_sub]
E_ref_on_mlip = np.array([ref_energy(fr_m[i]) for i in idx_sub])
traj_rmse = np.sqrt(np.mean(((E_mlip_sub - E_ref_on_mlip) / N_at)**2))
print(f"energy RMSE per atom along the MLIP run (every {every}th frame): {traj_rmse:.4e}")
print(f"drift of MLIP total energy: {(Ep_m + Ek_m)[-1] - (Ep_m + Ek_m)[0]:.4e}")
print(f"drift of ref  total energy: {(Ep_r + Ek_r)[-1] - (Ep_r + Ek_r)[0]:.4e}")

In [ ]:
# ---- check G1 ----
k = 0.5; harm = lambda X: (0.5 * k * np.sum(X**2), -k * X)        # 3D harmonic oscillators
Ep_h, Ek_h, fr_h = velocity_verlet(np.ones((2, 3)), harm, 0.5, 2000, dt=0.01)
Et_h = Ep_h + Ek_h
check(len(fr_h) == 2000 and Ep_h.shape == (2000,), "output lengths")
check(Et_h.std() / Et_h.mean() < 1e-3, f"harmonic test: relative energy fluctuation {Et_h.std()/Et_h.mean():.1e}")

**G2 comment:**

Both total energies stay flat: the MLIP drifts by about $2\times 10^{-5}$ and the reference by about $8\times 10^{-5}$ over the 1500 steps, so the integrator is conserving energy and the MLIP forces are coming from a scalar $E$. On every 10th MLIP frame the reference energy differs from the MLIP energy by an RMSE of $3.7\times 10^{-4}$ per atom, basically the same as the test-set error, so this short run does not leave the region the model was fit on.

---
## Task H. Extrapolation and a data-driven fix
**H1 (4 pts).** Scale the centred minimum structure `R_min` uniformly by $s\in[0.75,1.25]$ (25 values). Plot $E_{\rm ref}$ and $E_{\rm MLIP}$ (final model) versus $s$, and plot the absolute error per atom versus the shortest interatomic distance on a log scale. Mark the shortest distance present in the training set.

**H2 (6 pts).** Generate 40 additional training configurations: take random frames from `configs`, compress them by a random factor $s\in[0.75,0.95]$, and add Gaussian noise with standard deviation 0.03. Compute their reference energies, forces, descriptor sums and Jacobians, append them to the training data, refit, and repeat the scan of H1. Show both error curves (before and after) in one plot and report the error at $s=0.8$ for both models. Has the accuracy near equilibrium changed? Check with `rmse` on the original `test_idx`. Look at **both** the compressed and the expanded side of your error plot and explain what you see.

In [ ]:
def min_pair(R):
    _, rij = pair_geometry(R)
    np.fill_diagonal(rij, np.inf)
    return float(rij.min())

# training set = the 260 pool configurations the final model was fit on
r_train_min = min(min_pair(configs[i]) for i in pool_idx)
print(f"shortest distance in the training pool: {r_train_min:.4f}")

R_centred = R_min - R_min.mean(axis=0)
s_grid = np.linspace(0.75, 1.25, 25)

def energy_scan(model, s_values):
    E_ref = np.empty(len(s_values))
    E_hat = np.empty(len(s_values))
    rmin = np.empty(len(s_values))
    for i, s in enumerate(s_values):
        Rs = R_centred * s
        E_ref[i] = ref_energy(Rs)
        dsum, J = descriptor_sum_and_jacobian(Rs)
        E_hat[i], _ = predict(model, dsum, J)
        rmin[i] = min_pair(Rs)
    return E_ref, E_hat, rmin

Eref1, Ehat1, rmin1 = energy_scan(model_final, s_grid)
err1 = np.abs(Ehat1 - Eref1) / N_at

fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.0))
axes[0].plot(s_grid, Eref1, "o-", ms=4, label="reference")
axes[0].plot(s_grid, Ehat1, "s--", ms=4, label="MLIP")
axes[0].set_xlabel("scale s")
axes[0].set_ylabel("energy")
axes[0].set_title("Relaxed cluster scaled about its centre")
axes[0].legend()
axes[1].semilogy(rmin1, err1, "o-", ms=4)
axes[1].axvline(r_train_min, color="C3", ls="--", label="min training distance")
axes[1].set_xlabel("shortest pair distance")
axes[1].set_ylabel("|error| per atom")
axes[1].set_title("Error when the cluster is scaled")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

# H2 -- 40 compressed frames, refit, same scan
rng_h = np.random.default_rng(4)
chosen = rng_h.choice(len(configs), size=40, replace=False)
extra = []
for i in chosen:
    s = float(rng_h.uniform(0.75, 0.95))
    X = configs[i]
    c = X.mean(axis=0)
    Y = (X - c) * s + c
    Y = Y + rng_h.normal(0.0, 0.03, size=Y.shape)
    extra.append(Y)

print("reference energies/forces and Jacobians for the 40 extra frames")
extra_E, extra_F, extra_DS, extra_J = [], [], [], []
for Y in extra:
    extra_E.append(ref_energy(Y))
    extra_F.append(ref_forces(Y))
    dsum, J = descriptor_sum_and_jacobian(Y)
    extra_DS.append(dsum)
    extra_J.append(J)

DS_h = np.concatenate([DS, np.array(extra_DS)], axis=0)
JAC_h = np.concatenate([JAC, np.array(extra_J)], axis=0)
E_h = np.concatenate([energies, np.array(extra_E)])
F_h = np.concatenate([forces, np.array(extra_F)])
new_idx = np.arange(len(configs), len(E_h))
train_h = np.concatenate([pool_idx, new_idx])

model_h2 = fit_linear_mlip(train_h, cols_all, use_forces=True,
                           DS=DS_h, JAC=JAC_h, energies=E_h, forces=F_h)

Eref2, Ehat2, rmin2 = energy_scan(model_h2, s_grid)
err2 = np.abs(Ehat2 - Eref2) / N_at

fig, ax = plt.subplots(figsize=(6.4, 4.1))
ax.semilogy(rmin1, err1, "o-", ms=4, label="before")
ax.semilogy(rmin2, err2, "s-", ms=4, label="after extra data")
ax.axvline(r_train_min, color="C3", ls="--", label="min training distance")
ax.set_xlabel("shortest pair distance")
ax.set_ylabel("|error| per atom")
ax.set_title("Same scaling scan, before and after")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

# exactly s = 0.8, not just the nearest grid point
Rs08 = R_centred * 0.8
d08, J08 = descriptor_sum_and_jacobian(Rs08)
E08_ref = ref_energy(Rs08)
E08_before, _ = predict(model_final, d08, J08)
E08_after, _ = predict(model_h2, d08, J08)
err08_before = abs(E08_before - E08_ref) / N_at
err08_after = abs(E08_after - E08_ref) / N_at
print(f"|error| per atom at s = 0.8, before: {err08_before:.4e}")
print(f"|error| per atom at s = 0.8, after : {err08_after:.4e}")

e0, f0 = rmse(model_final, test_idx)
e1, f1 = rmse(model_h2, test_idx)
print(f"test set before: E = {e0:.4e} per atom, F = {f0:.4f}")
print(f"test set after : E = {e1:.4e} per atom, F = {f1:.4f}")
print(f"most compressed point s = {s_grid[0]:.3f}, error before/after = {err1[0]:.4e} / {err2[0]:.4e}")
print(f"most expanded point  s = {s_grid[-1]:.3f}, error before/after = {err1[-1]:.4e} / {err2[-1]:.4e}")

**H2 comment:**

At $s=0.8$ the absolute error per atom drops from $2.2\times 10^{-2}$ to $1.3\times 10^{-3}$. On the original test set the energy RMSE goes from $2.8\times 10^{-4}$ to $5.2\times 10^{-4}$ per atom and the force RMSE from 0.009 to 0.016, so near equilibrium the model is a bit worse but still accurate. The short-distance side of the plot is where the new data helps (at $s=0.75$ the error falls from $5.7\times 10^{-2}$ to $1.4\times 10^{-3}$). The stretched side gets worse, from $1.7\times 10^{-4}$ to $9.4\times 10^{-3}$ at $s=1.25$, because those 40 frames were only compressed, so the refit spends parameters on short bonds and gives some of the long-bond accuracy back.

---
## Task I. Written questions
Answer briefly (three to five sentences each).

1. The atomic energy $E_i$ in Eq. $E=\sum_iE_i$ is not a quantum-mechanical observable. Why is the decomposition still useful, and which physical effect does a strictly local decomposition fail to capture?
2. Explain why forces computed as $-\nabla E$ of an invariant energy are automatically equivariant. Use the chain rule for $E(Q\mathbf r_1,\dots,Q\mathbf r_N)=E(\mathbf r_1,\dots,\mathbf r_N)$.
3. A message-passing model has $T=3$ layers and $r_c=5$ Å. What is its receptive field? Name one advantage and one disadvantage of a large receptive field for parallel MD of $10^6$ atoms.
4. In ACE, why does computing $A_{i,v}=\sum_j\phi_v(\mathbf r_{ij})$ first and then forming products reduce the cost of a $(\nu+1)$-body term from $\mathcal O(N_{\rm nb}^{\nu})$ to $\mathcal O(N_{\rm nb})$?
5. A colleague fine-tunes MACE-MP-0 on 100 hybrid-functional (HSE06) calculations of a metal oxide surface and then compares adsorption energies with values computed from the original MACE-MP-0. Identify two pitfalls.

**Your answers:**

1. $E_i$ is only a way of writing the total energy, not a number you can measure on one atom. The split is still useful because the same local function then works for any number of atoms, and the cost follows the neighbour list instead of the whole box. A hard cutoff misses interactions that reach past $r_c$. The obvious one is electrostatics: charge transfer and the Coulomb tail are not determined by the atoms inside a small sphere.

2. Invariance says $E(Q\mathbf r_1,\ldots,Q\mathbf r_N) = E(\mathbf r_1,\ldots,\mathbf r_N)$. Write $\mathbf r'_j = Q\mathbf r_j$ and differentiate both sides in $r_{i\alpha}$. The right-hand side is just $\partial E/\partial r_{i\alpha}$. On the left the chain rule multiplies by $\partial r'_{i\beta}/\partial r_{i\alpha} = Q_{\beta\alpha}$, so $\nabla_i E(\mathbf r) = Q^{\mathsf T}\nabla'_i E(\mathbf r')$. The force is minus that gradient, which rearranges to $\mathbf F_i(\mathbf r') = Q\mathbf F_i(\mathbf r)$. The force is rotated by the same $Q$ as the coordinates. No extra term shows up, because the value of $E$ did not change, only the frame did.

3. Each layer adds one cutoff, so the receptive field is $T r_c = 15$ Å. An atom can feel other atoms up to 15 Å away even though messages only travel 5 Å per layer. That helps when the physics is not that local, for example a surface or a medium-range angular pattern. For a parallel run of $10^6$ atoms the downside is the halo: every domain has to import a 15 Å shell of ghost atoms, and the neighbour list and the communication both grow quickly.

4. $A_{i,v} = \sum_j \phi_v(\mathbf r_{ij})$ is a single loop over neighbours. A product of $\nu$ of these $A$'s expands into a sum over $\nu$ neighbour indices, so the $(\nu+1)$-body combinations are already inside the product. You never write the explicit loop over tuples, which would have been $\mathcal O(N_{\rm nb}^{\nu})$. Building $A$ is $\mathcal O(N_{\rm nb})$, and the products only depend on how many basis functions you kept.

5. One hundred HSE06 slabs is a very small fine-tuning set, so the fit can lock onto that particular surface and forget configurations that MACE-MP-0 previously got right. The second problem is the comparison itself. Checking the new adsorption energies against the original MACE-MP-0 does not show that you moved towards HSE06. The base model was trained mostly on PBE, so the two models can share the same error, and the adsorption number also depends on using the same slab, the same isolated molecule and the same relaxation.

---
## Bonus. A power-spectrum (SOAP-like) descriptor

Replace the 16 $G^4$ functions by the 3-body power spectrum written through the addition theorem (§12 of the notes):

$$
p_{i,nl}=\frac{2l+1}{4\pi}\sum_{j\ne i}\sum_{k\ne i}R_n(r_{ij})\,R_n(r_{ik})\,P_l(\cos\theta_{jik}),\qquad R_n(r)=e^{-\eta_n r^2}f_c(r),
$$

with $l=0,1,2,3$, Legendre polynomials $P_l$, and four radial functions $\eta_n\in\{0.3,0.8,1.5,3.0\}$ (16 features in total). Note that the double sum includes $j=k$, as in ACE. Build the descriptor `G2 + power spectrum`, compute the Jacobians, fit an E+F model on the full pool, and compare its test errors with G2+G4. Verify numerically that $p_{i,n1}=|\mathbf A_{i,n1}|^2$ for one atom, where $\mathbf A_{i,n1}=\sqrt{3/4\pi}\sum_jR_n(r_{ij})\hat{\mathbf r}_{ij}$.